In [1]:
import urllib.error
import urllib.request
import pprint
from langchain.tools import tool

from langchain.chat_models import init_chat_model

import langchain_groq
import os

from dotenv import load_dotenv

load_dotenv("C:\\Users\\socgen\\ML\\agentic_ai_and_ops\\langchain_day5\\.env")

True

In [2]:

model_gr_lamma = init_chat_model("llama-3.3-70b-versatile",
                        api_key=os.environ["GROQ_API_KEY"],
                        model_provider="groq",
                        # base_url="https://api.groq.com/openai/v1",
                        max_tokens=1000, temperature=0.0)

model_or_paid_gpt40 = init_chat_model("openai/gpt-4o-mini",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

model_or_free_nvidia = init_chat_model("nvidia/nemotron-3-ultra-550b-a55b:free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)


model_or_free = init_chat_model("openrouter/free",
                        api_key=os.environ["OPENROUTER_API_KEY"],
                        model_provider="openrouter",
                        base_url="https://openrouter.ai/api/v1",
                        max_tokens=1000, temperature=0.0)

## Sample agent

In [8]:
from langchain.agents import create_agent
from langchain.tools import tool


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

In [9]:
agent = create_agent(model=model_gr_lamma, tools=[search])

In [10]:
agent.get_graph()

Graph(nodes={'__start__': Node(id='__start__', name='__start__', data=RunnableCallable(tags=None, recurse=True, explode_args=False, func_accepts={}), metadata=None), 'model': Node(id='model', name='model', data=model_node(tags=None, recurse=True, explode_args=False, func_accepts={'runtime': ('N/A', <class 'inspect._empty'>)}), metadata=None), 'tools': Node(id='tools', name='tools', data=tools(tags=None, recurse=True, explode_args=False, func_accepts={'config': ('N/A', <class 'inspect._empty'>), 'runtime': ('N/A', <class 'inspect._empty'>)}, _tools_by_name={'search': StructuredTool(name='search', description='Search for information.', args_schema=<class 'langchain_core.utils.pydantic.search'>, func=<function search at 0x000002B0B0EC5120>)}, _injected_args={'search': _InjectedArgs(state={}, store=None, runtime=None, all_injected_keys=set(), _optional_state_args=set())}, _handle_tool_errors=<function _default_handle_tool_errors at 0x000002B0B0E56520>, _messages_key='messages', _wrap_tool_

# Real tools

In [11]:
import requests

@tool
def get_real_weather(city: str) -> str:
    """Get the REAL current weather for a city, using a live weather API."""
    # Step 1: geocode the city name into latitude/longitude
    geo_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
    )
    geo_data = geo_response.json()
    if not geo_data.get("results"):
        return f"Could not find a location matching '{city}'."

    location = geo_data["results"][0]
    lat, lon = location["latitude"], location["longitude"]

    # Step 2: fetch the real current weather at that exact location
    weather_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={"latitude": lat, "longitude": lon, "current_weather": "true"},
    )
    weather_data = weather_response.json()
    current = weather_data.get("current_weather", {})

    return (
        f"Real current weather in {location['name']}, {location.get('country', '')}: "
        f"{current.get('temperature')}°C, wind {current.get('windspeed')} km/h."
    )

print("Tool defined -- run the cell below to hit the LIVE API for real.")

Tool defined -- run the cell below to hit the LIVE API for real.


In [12]:
os.environ["TAVILY_API_KEY"]

'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'

In [48]:
from langchain_tavily import TavilySearch
tavily_search = TavilySearch(max_results=5, topic="general")

In [13]:
import sqlite3

In [16]:
def setup_database():
  conn = sqlite3.connect("tripmate.db")
  cur = conn.cursor()
  cur.execute("""
    CREATE TABLE IF NOT EXISTS trips (
        trip_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT,
        destination TEXT,
        start_date TEXT,
        end_date TEXT,
        status TEXT DEFAULT 'confirmed'
    )
""")
  conn.commit()
  conn.close()

In [17]:
setup_database()

In [18]:
@tool
def save_trip(user_id: str, destination: str, start_date: str, end_date: str) -> str:
    """Save a new trip to the real database."""
    conn = sqlite3.connect("tripmate.db")
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO trips (user_id, destination, start_date, end_date) VALUES (?, ?, ?, ?)",
        (user_id, destination, start_date, end_date),
    )
    conn.commit()
    trip_id = cur.lastrowid
    conn.close()
    return f"Trip #{trip_id} saved: {destination}, {start_date} to {end_date}."

@tool
def get_saved_trips(user_id: str) -> str:
    """Look up all saved trips for a user from the real database."""
    conn = sqlite3.connect("tripmate.db")
    cur = conn.cursor()
    cur.execute("SELECT trip_id, destination, start_date, end_date, status FROM trips WHERE user_id = ?", (user_id,))
    rows = cur.fetchall()
    conn.close()
    if not rows:
        return "No saved trips found for this user."
    return "\n".join(f"Trip #{r[0]}: {r[1]} ({r[2]} to {r[3]}) -- {r[4]}" for r in rows)

In [21]:
# Prove this is REAL persistence -- save, then read back
print(save_trip.invoke({"user_id": "rohan_01", "destination": "Bali", "start_date": "2026-09-01", "end_date": "2026-09-10"}))


Trip #2 saved: Bali, 2026-09-01 to 2026-09-10.


In [22]:
print(get_saved_trips.invoke({"user_id": "rohan_01"}))

Trip #1: Bali (2026-09-01 to 2026-09-10) -- confirmed
Trip #2: Bali (2026-09-01 to 2026-09-10) -- confirmed


## Detour to understand stateful agent with checkpointer and in memory saver

In [23]:
from langchain_core.tools import tool


@tool
def check_showtimes(movie: str, city: str) -> str:
    """
    Check available showtimes for a movie in a given city.
    """
    return f"""
Available showtimes for '{movie}' in {city}:

🎬 PVR IMAX
- 10:00 AM
- 1:30 PM
- 5:00 PM
- 8:30 PM

🎬 Cinepolis
- 11:15 AM
- 2:45 PM
- 6:15 PM
- 9:45 PM

🎬 INOX
- 9:45 AM
- 12:30 PM
- 4:00 PM
- 7:30 PM

Seats are currently available for all shows.
"""


@tool
def book_seats(
    movie: str,
    theatre: str,
    showtime: str,
    seats: int,
) -> str:
    """
    Book seats for a movie show.
    """
    booking_id = "BK123456"

    return (
        f"✅ Booking Confirmed!\n\n"
        f"Movie: {movie}\n"
        f"Theatre: {theatre}\n"
        f"Showtime: {showtime}\n"
        f"Seats Booked: {seats}\n"
        f"Booking ID: {booking_id}\n"
        f"Status: Confirmed\n\n"
        f"Enjoy your movie!"
    )


@tool
def get_exact_refund_policy() -> str:
    """
    Return the cinema's refund and cancellation policy.
    """
    return """
🎟️ Cinema Refund & Cancellation Policy

• Tickets can be cancelled up to 2 hours before the showtime.
• A cancellation fee of 10% of the ticket amount will be deducted.
• No refunds are allowed within 2 hours of the show.
• Convenience fees are non-refundable.
• Refunds are credited to the original payment method within 5–7 business days.
• Tickets purchased using promotional offers or coupons are not eligible for a full refund.
• In case the show is cancelled by the cinema, a full refund (including convenience fee) will be issued automatically.
"""


# Register these with your agent
tools = [
    check_showtimes,
    book_seats,
    get_exact_refund_policy,
]

In [25]:
cinebot = create_agent(
    model=model_gr_lamma,
    tools=[check_showtimes, book_seats, get_exact_refund_policy],
    system_prompt="You are CineBot, a friendly cinema booking assistant. Check showtimes before booking.",
)

In [27]:
cinebot.invoke({"messages":[('user','My Name is Sunj')]})

{'messages': [HumanMessage(content='My Name is Sunj', additional_kwargs={}, response_metadata={}, id='2b5f48a9-2e06-4e9b-ad13-a27fb34cde1a'),
  AIMessage(content='Hello Sunj, welcome to CineBot. What can I help you with today? Would you like to check showtimes for a movie or book seats?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 390, 'total_tokens': 423, 'completion_time': 0.080398383, 'completion_tokens_details': None, 'prompt_time': 0.019767881, 'prompt_tokens_details': None, 'queue_time': 0.162542127, 'total_time': 0.100166264}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fbbaf-3222-7df2-b24c-e7fd442c2037-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 390, 'output_tokens': 33, 'total_tokens': 423})]}

In [28]:
cinebot.invoke({"messages":[('user','Who am I ?')]})

{'messages': [HumanMessage(content='Who am I ?', additional_kwargs={}, response_metadata={}, id='d2e534bd-64a0-4a03-b773-4fff8703b920'),
  AIMessage(content='You are the user seeking assistance from CineBot, a friendly cinema booking assistant.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 389, 'total_tokens': 407, 'completion_time': 0.056408694, 'completion_tokens_details': None, 'prompt_time': 0.036953715, 'prompt_tokens_details': None, 'queue_time': 0.056193414, 'total_time': 0.093362409}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fbbaf-772f-7781-96d9-d19b95a77da4-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 389, 'output_tokens': 18, 'total_tokens': 407})]}

# InMemorySaver or Checkpointing

In [37]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

In [38]:
checkpointer = InMemorySaver()

config: RunnableConfig = {"configurable": {"thread_id": "1"}}

In [39]:
cinebot_with_chkpoint = create_agent(
    model=model_gr_lamma,
    tools=[check_showtimes, book_seats, get_exact_refund_policy],
    system_prompt="You are CineBot, a friendly cinema booking assistant. Check showtimes before booking.",
    checkpointer=checkpointer,
)

In [40]:
cinebot_with_chkpoint.invoke({"messages":[('user','My Name is Sunj')]},config=config)

{'messages': [HumanMessage(content='My Name is Sunj', additional_kwargs={}, response_metadata={}, id='d6182425-e439-414d-92d4-195ed6f7d9b8'),
  AIMessage(content='Hello Sunj, welcome to CineBot. What can I help you with today? Would you like to check showtimes for a movie or book seats?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 390, 'total_tokens': 423, 'completion_time': 0.086993994, 'completion_tokens_details': None, 'prompt_time': 0.019766331, 'prompt_tokens_details': None, 'queue_time': 0.161719937, 'total_time': 0.106760325}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fbbb4-2456-73f0-b591-445b5e8d6d32-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 390, 'output_tokens': 33, 'total_tokens': 423})]}

In [41]:
cinebot_with_chkpoint.invoke({"messages":[('user','Who am I ?')]},config=config)

{'messages': [HumanMessage(content='My Name is Sunj', additional_kwargs={}, response_metadata={}, id='d6182425-e439-414d-92d4-195ed6f7d9b8'),
  AIMessage(content='Hello Sunj, welcome to CineBot. What can I help you with today? Would you like to check showtimes for a movie or book seats?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 390, 'total_tokens': 423, 'completion_time': 0.086993994, 'completion_tokens_details': None, 'prompt_time': 0.019766331, 'prompt_tokens_details': None, 'queue_time': 0.161719937, 'total_time': 0.106760325}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fbbb4-2456-73f0-b591-445b5e8d6d32-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 390, 'output_tokens': 33, 'total_tokens': 423}),
  HumanMessage(content='Who am I ?', additional_kwarg

In [42]:
new_config = {"configurable": {"thread_id": "2"}}

In [43]:
cinebot_with_chkpoint.invoke({"messages":[('user','Who am I ?')]},config=new_config)

{'messages': [HumanMessage(content='Who am I ?', additional_kwargs={}, response_metadata={}, id='b1912693-cf29-43cb-ae9f-3334adf346d5'),
  AIMessage(content='You are the user of the CineBot, a friendly cinema booking assistant.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 389, 'total_tokens': 406, 'completion_time': 0.055955792, 'completion_tokens_details': None, 'prompt_time': 0.033520945, 'prompt_tokens_details': None, 'queue_time': 0.054398054, 'total_time': 0.089476737}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fbbb9-82e6-7b12-aae3-2da9ae4819de-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 389, 'output_tokens': 17, 'total_tokens': 406})]}

## back to tripmate agent

In [44]:
trip_checkpointer = InMemorySaver()

trip_config: RunnableConfig = {"configurable": {"thread_id": "1"}}

In [49]:
tripmate_with_chkpoint = create_agent(
    model=model_gr_lamma,
    tools=[get_real_weather, tavily_search, save_trip, get_saved_trips],
    system_prompt="You are TripMate, a helpful travel assistant. Provide accurate information about destinations and activities.",
    checkpointer=trip_checkpointer,
)

In [50]:
response= tripmate_with_chkpoint.invoke({"messages":[('user','Iam Sunj, Kindly book a trip to Paris from 10 Aug 2026 to 15 Aug 2026')]},config=trip_config)

In [51]:
import pprint

In [52]:
pprint.pprint(response)

{'messages': [HumanMessage(content='Iam Sunj, Kindly book a trip to Paris from 10 Aug 2026 to 15 Aug 2026', additional_kwargs={}, response_metadata={}, id='bcdb443c-810b-40d0-824d-6f1fbcdfc5d0'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '7m7vyrmzh', 'function': {'arguments': '{"destination":"Paris","end_date":"2026-08-15","start_date":"2026-08-10","user_id":"Sunj"}', 'name': 'save_trip'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 1949, 'total_tokens': 1996, 'completion_time': 0.119550334, 'completion_tokens_details': None, 'prompt_time': 0.117125683, 'prompt_tokens_details': None, 'queue_time': 0.051866465, 'total_time': 0.236676017}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fbbc5-e01e-7a33-8623-6587287bb206-0', tool_calls=[{'name': 'sav

### Long Term Memory via InMemoryStore

In [53]:
from typing import Any
from langgraph.store.memory import InMemoryStore

travel_store = InMemoryStore()

In [54]:
from langchain.tools import ToolRuntime
@tool
def save_travel_style(user_id: str, style: str, runtime: ToolRuntime) -> str:
    """Save a traveler's preferred trip style (e.g. budget, luxury, adventure) for future visits."""
    runtime.store.put((user_id, "preferences"), "travel_style", {"value": style})
    return f"Noted -- I'll remember you prefer {style} travel."

@tool
def recall_travel_style(user_id: str, runtime: ToolRuntime) -> str:
    """Recall a traveler's preferred trip style, if saved before."""
    result = runtime.store.get((user_id, "preferences"), "travel_style")
    return result.value["value"] if result else "No travel style saved yet for this user."


In [55]:

print("Tools defined -- runtime.store is a genuinely separate memory system from the")
print("SQLite database above. Trips are structured records; preferences are lightweight facts.")


Tools defined -- runtime.store is a genuinely separate memory system from the
SQLite database above. Trips are structured records; preferences are lightweight facts.


## Dynamic tool gating

In [56]:
@tool
def book_premium_concierge(destination: str) -> str:
    """Book a dedicated human concierge for trip planning. Premium members only."""
    return f"Premium concierge assigned for your {destination} trip."

In [58]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

In [ ]:
@wrap_model_call
def gate_premium_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Only expose book_premium_concierge to premium members."""
    is_premium = request.state.get("is_premium_member", False)
    if not is_premium:
        allowed = [t for t in request.tools if t.name != "book_premium_concierge"]
        request = request.override(tools=allowed)
    return handler(request)

In [71]:

from dataclasses import dataclass
@dataclass
class TravelerContext:
    user_id: str
    home_currency: str
    membership_tier: str
    is_premium_member :bool =False

In [63]:
from langchain.agents.structured_output import ProviderStrategy, ToolStrategy

In [64]:
from pydantic import BaseModel, Field
from typing import Union, Literal
from langchain.agents.structured_output import ToolStrategy

class NewTripRequest(BaseModel):
    """A request to plan a new trip."""
    user_id: str
    destination: str
    start_date: str = Field(description="Format: YYYY-MM-DD")
    end_date: str = Field(description="Format: YYYY-MM-DD")

class ModifyTripRequest(BaseModel):
    """A request to change an existing trip."""
    user_id: str
    trip_id: int
    change_description: str

class CancelTripRequest(BaseModel):
    """A request to cancel an existing trip."""
    user_id: str
    trip_id: int

In [76]:
model_ollama = init_chat_model("ollama:gemma4")

In [77]:
full_tripmate = create_agent(
    model=model_ollama,
    tools=[
        tavily_search, # web search for destinations and activities
        get_real_weather, # real weather api
        save_trip, # save a trip to the database sqlite
        get_saved_trips, # retrieve saved trips from the database sqlite
        save_travel_style, # save a traveler's preferred trip style (e.g. budget, luxury, adventure) for future visits in in memory store
        recall_travel_style, # recall a traveler's preferred trip style, if saved before in memory store
        book_premium_concierge, # book a dedicated human concierge for trip planning. Premium members only.
    ],
    system_prompt=(
        "You are TripMate, a real travel planning assistant. Check real weather before "
        "recommending destinations. Save trips when confirmed. Remember travel style preferences."
    ),
    middleware=[gate_premium_tools], # middleware to gate premium tools based on membership tier
    checkpointer=InMemorySaver(), # checkpointer to save the agent's state in memory for this session stateful agent
    store=travel_store, # store for long-term memory of traveler's preferences
    context_schema=TravelerContext, # schema for the traveler's context
    name="tripmate_agent",
    response_format=ToolStrategy(Union[NewTripRequest, ModifyTripRequest, CancelTripRequest])

)

In [78]:
config = {"configurable": {"thread_id": "Sundara-planning-session"}}

In [79]:
result = full_tripmate.invoke(
    {
        "messages": [("user", "I'm Sundara. What's the weather like in Bali? I prefer budget travel, please remember that.")],
        "is_premium_member": False,
    },
    config=config,
    context=TravelerContext(user_id="Sundara", home_currency="INR", membership_tier="standard",is_premium_member=False),
)

c:\Users\socgen\ML\agentic_ai_and_ops\.venv\Lib\site-packages\pydantic\functional_validators.py:835: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=TravelerContext(user_id='...is_premium_member=False), input_type=TravelerContext])
  function=lambda v, h: h(v), schema=original_schema
c:\Users\socgen\ML\agentic_ai_and_ops\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=TravelerContext(user_id='...is_premium_member=False), input_type=TravelerContext])
  return self.__pydantic_serializer__.to_python(


In [80]:
result

{'messages': [HumanMessage(content="I'm Sundara. What's the weather like in Bali? I prefer budget travel, please remember that.", additional_kwargs={}, response_metadata={}, id='5b57bfed-393b-44c9-8547-b83bfaa7483c'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'gemma4', 'created_at': '2026-08-01T06:10:44.2260922Z', 'done': True, 'done_reason': 'stop', 'total_duration': 21802370500, 'load_duration': 14535966000, 'prompt_eval_count': 2015, 'prompt_eval_duration': 1651850000, 'eval_count': 196, 'eval_duration': 5572790000, 'logprobs': None, 'model_name': 'gemma4', 'model_provider': 'ollama'}, name='tripmate_agent', id='lc_run--019fbbf1-ee55-77f0-8db2-c2999c3e14b7-0', tool_calls=[{'name': 'get_real_weather', 'args': {'city': 'Bali'}, 'id': '0b9f5a2c-e10f-4984-a1e8-369a14b5ce4c', 'type': 'tool_call'}, {'name': 'save_travel_style', 'args': {'style': 'budget', 'user_id': 'Sundara'}, 'id': '8fcbcd43-2fc0-4ef0-a7c6-2df76263ef86', 'type': 'tool_call'}], invalid_too